# 02 - Polygon 2-Day Intraday Profile

Pulls 1-minute bars for top movers and profiles volatility/volume behavior.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
import plotly.express as px

from src.common import (
    load_env, get_api_keys, load_config, new_pull_id, polygon_gainers,
    polygon_intraday_bars, save_raw_json, save_parquet
)

load_env('../.env')
cfg = load_config('../config/exploration.yaml')
keys = get_api_keys()
assert keys['polygon'], 'POLYGON_API_KEY missing in data-exploration/.env'


In [ ]:
top_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
tickers = [x.get('ticker') for x in top_payload.get('tickers', []) if x.get('ticker')][:25]

end = datetime.now(tz=ZoneInfo('UTC')).date()
start = end - timedelta(days=4)
start_date = start.isoformat()
end_date = end.isoformat()

pull_id = new_pull_id('polygon_intraday')
all_rows = []

for symbol in tickers:
    bars = polygon_intraday_bars(keys['polygon'], keys['polygon_base'], symbol, start_date, end_date)
    save_raw_json(bars, cfg.outputs_raw_dir / f'{pull_id}_{symbol}_bars.json')
    for r in bars.get('results', []) or []:
        all_rows.append({
            'provider': 'polygon',
            'symbol': symbol,
            'ts_utc': pd.to_datetime(r['t'], unit='ms', utc=True),
            'open': r.get('o'),
            'high': r.get('h'),
            'low': r.get('l'),
            'close': r.get('c'),
            'volume': r.get('v'),
            'source_endpoint': '/v2/aggs/ticker/{symbol}/range/1/minute/{from}/{to}',
            'pull_id': pull_id,
        })

df_intraday = pd.DataFrame(all_rows).sort_values(['symbol','ts_utc'])
if not df_intraday.empty:
    df_intraday['ret_1m_pct'] = df_intraday.groupby('symbol')['close'].pct_change() * 100
    df_intraday['rv_20m'] = df_intraday.groupby('symbol')['ret_1m_pct'].rolling(20).std().reset_index(level=0, drop=True)

save_parquet(df_intraday, cfg.outputs_processed_dir / f'{pull_id}_polygon_intraday_2d.parquet')
df_intraday.head()


In [ ]:
if not df_intraday.empty:
    prof = df_intraday.groupby('symbol').agg(
        bars=('symbol','size'),
        avg_volume=('volume','mean'),
        p95_abs_ret=('ret_1m_pct', lambda s: s.abs().quantile(0.95)),
        avg_rv_20m=('rv_20m','mean'),
    ).sort_values('p95_abs_ret', ascending=False)
    display(prof.head(20))


In [ ]:
if not df_intraday.empty:
    fig = px.box(df_intraday, x='symbol', y='ret_1m_pct', title='1-Min Return Distribution by Symbol')
    fig.show()
